In [1]:
import pandas as pd
import numpy as np

### Load the data

In [2]:
cubo = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\cubo_1.xlsx", 
                    sheet_name = 'SKUs',
                    skiprows=4
                    )

In [ ]:
ventas = 'C:\\Users\\fdavila\\OneDrive - Farmacorp S.A\\Escritorio\\cross selling\\data\\DATOS DE VENTA DE ENE-MAR 2025.xlsx'

In [ ]:
# Crear una lista para almacenar los DataFrames
dataframes = []

# Leer los 10 sheets
for i in range(1, 11):
    nombre_sheet = f'Sheet {i}'
    df = pd.read_excel(ventas, sheet_name=nombre_sheet)
    dataframes.append(df)
    print(f'Leído {nombre_sheet}: {len(df)} filas')

# Concatenar todos los DataFrames
df_combinado = pd.concat(dataframes, ignore_index=True)

# Exportar a CSV
df_combinado.to_csv('resultado_combinado.csv', index=False)

print(f'\n✓ Archivo CSV creado con {len(df_combinado)} filas totales')

In [3]:
ventasfarma = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\ventas_01_03-25.csv")

In [6]:
ventasamkt = pd.read_csv(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\cross selling\data\DATOS DE VENTA DE ENE-MAR_ 2025 amarke.csv", sep=";")

In [4]:
ventasfarma.shape

(9987646, 6)

In [7]:
ventasamkt.shape

(2214604, 6)

Contamos con el registro de 2,214,604 facturas en la UNE FARMACORP
Contamos con el registro de 9,987,646 facturas en la UNE AMARKET

In [8]:
clusters = pd.read_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\data\clusters_bdgs.xlsx")

In [9]:
#convertimos fechas a formato datetime
ventasfarma['FECHA_FACT']= pd.to_datetime(ventasfarma['FECHA_FACT'])

In [10]:
#visualozamos las ventas por dia
ventasxdia = ventasfarma.FECHA_FACT.value_counts().reset_index()

In [13]:
ventasxdia

,FECHA_FACT,count
0,2025-03-01,133659
1,2025-02-28,128955
2,2025-01-02,128305
3,2025-03-05,127125
4,2025-03-31,126489
...,...,...
85,2025-01-30,101507
86,2025-03-03,98300
87,2025-01-29,97832
88,2025-03-04,94747


In [14]:
#vamos a filtrar solo BODEGAS FARMACORP para este analisis
clusters_fc = clusters[clusters['UNE']=='FARMACORP']

In [15]:
ventasxdia.columns = ['FECHA_FACT', 'CANT_VENTAS']

In [16]:
ventasxdia

,FECHA_FACT,CANT_VENTAS
0,2025-03-01,133659
1,2025-02-28,128955
2,2025-01-02,128305
3,2025-03-05,127125
4,2025-03-31,126489
...,...,...
85,2025-01-30,101507
86,2025-03-03,98300
87,2025-01-29,97832
88,2025-03-04,94747


El dia que hubo mas facturacion en UNE FARMACORP fue el 2022-03-01 con un total de 133,659 facturas, esto ocurre un sabado antes del feriado de Carnavales

In [17]:
ventasxfact = ventasfarma.copy()

In [ ]:
#analizamos el dia com mas movimiento (SABADO antes de Carnavales)
#ventasxfact = ventasfarma[(ventasfarma['FECHA_FACT'] == '2025-03-01 00:00:00')]

In [18]:
#analizamos los dias de carnavales
ventasxfact = ventasxfact[(ventasxfact['FECHA_FACT'] >= '2025-03-01 00:00:00') & (ventasxfact['FECHA_FACT'] <= '2025-03-05 00:00:00')]

In [19]:
ventasxfact.shape

(563430, 6)

### Procesamos la data

In [20]:
cubo['COD_ARTICULO'] = cubo['COD_ARTICULO'].astype(str)
ventasxfact['COD_ARTICULO'] = ventasxfact['COD_ARTICULO'].astype(str)

In [21]:
# filtering only SKUs on FARMACORP
ventas_farmacia = pd.merge(cubo,
                            ventasxfact,
                            on='COD_ARTICULO',
                            how='inner'
                            )

In [22]:
#base sobre la cual se va a trabajar
ventas_farmacia = pd.merge(
    ventas_farmacia,
    clusters_fc,
    left_on='COD_BODEGA',
    right_on='BODEGA',
)

In [23]:
ventas_farmacia.shape

(550446, 24)

### Calculamos medidas

In [24]:
ventas_farmacia.columns

Index(['CAT 0', 'CAT 1', 'CAT 2', 'CAT 3', 'CAT 4', 'COD_ARTICULO', 'ARTICULO',
       'Precio Unitario FA', 'CR Unitario', 'FECHA_FACT', 'COD_BODEGA',
       'NUMERO_FACTURA', 'UNIDADES', 'VENTA_NETA', 'BODEGA', 'UNE', 'BODEGA2',
       'CIUDAD', 'REGION', 'FORMATO', 'NSE', 'CLUSTER', 'BEAUTY',
       'HOSPITALARIA'],
      dtype='object')

In [25]:
#calculamos el total de facturas emitidas en el rango de fechas de los datos
total_fact = ventas_farmacia['NUMERO_FACTURA'].nunique()
total_fact

274564

In [26]:
#contamos facturas en las que aparece al menos 1 vez cada SKU
ventas_sku = ventas_farmacia.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NUMERO_FACTURA'].nunique().reset_index().sort_values(by='NUMERO_FACTURA', ascending=False)

In [27]:
#calculamos el peso de cada SKU sobre el total de las facturas
ventas_sku['weight'] = (ventas_sku['NUMERO_FACTURA']/total_fact).round(8)

In [28]:
#seleccionamos las facturas que solo tienen 1 sku
facturas_unicas = (
    ventas_farmacia
    .groupby(['NUMERO_FACTURA'])
    .size()
    .loc[lambda x: x == 1]
    .reset_index()[['NUMERO_FACTURA']]
)

#filtramos la data con los sku que aparecen solos en 1 factura
ventas_solo = ventas_farmacia.merge(
    facturas_unicas,
    on=['NUMERO_FACTURA'],
    how='inner'
)

In [29]:
ventas_solo['CAT 0'].value_counts()

CAT 0
ETICOS              53742
OTC                 37698
BEBIDAS             10999
CUIDADO PERSONAL     8812
INSUMOS MEDICOS      6494
IMPULSO              6308
CUIDADO INFANTIL     5633
PERECEDEROS          4588
HOGAR                3843
ABARROTES             850
Name: count, dtype: int64

Se encuentra una oportunidad de venta cruzada en las categorias ETICOS y OTC

In [32]:
#contamos facturas con un solo sku
total_fact_solo = ventas_solo['NUMERO_FACTURA'].nunique()
total_fact_solo

138967

In [33]:
#calculamos el % que representa del total de facturas
fact_solo = total_fact_solo/total_fact
fact_solo

0.5061370026660451

In [37]:
#contamos cuantas veces aparece cada sku solo 
ventas_sku_solo = ventas_solo.groupby(
    ['COD_ARTICULO', 'ARTICULO']
    )['NUMERO_FACTURA'].count().reset_index().sort_values(by='NUMERO_FACTURA', ascending=False)

In [38]:
ventas_sku_solo

,COD_ARTICULO,ARTICULO,NUMERO_FACTURA
4245,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),1578
4888,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,1557
4938,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,995
5394,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),915
1342,255701,MIGRANOL X 100 COMP V+,795
...,...,...,...
3746,7595751003378,APEX REVO ESPUMA P/AFEITAR X 414ML CLASSIC<DESC>,1
3742,75916909,HEINZ CRECIDITOS X 113GR MELOCOTON/DURAZNO<DESC>,1
3741,75916091,HEINZ CRECIDITOS X 113GR PERA (CAJX24)<DESC>,1
3739,75916060,HEINZ CRECIDITOS X 113GR FRUTAS TROPICALES<DESC>,1


El SKU que se vende mas solo es ZOPICLONA, seguido de QUETOROL, RESSAKA y TYPIREC

In [39]:
#calculamos el peso de cada sku en las facturas solas
ventas_sku_solo['weight'] = ventas_sku_solo['NUMERO_FACTURA']/total_fact_solo

In [48]:
#unimos todo
skus_analysis = pd.merge(
    ventas_sku,
    ventas_sku_solo,
    on='COD_ARTICULO',
    how='left',
    suffixes=['_gral', '_alone']
)

In [49]:
#calculamos el % en peso de las veces que cada sku aparece solo vs el total de veces que aparece en una factura
skus_analysis['weight_overeach'] = skus_analysis['NUMERO_FACTURA_alone']/skus_analysis['NUMERO_FACTURA_gral']

In [50]:
skus_analysis['weight_overall'] = skus_analysis['NUMERO_FACTURA_alone']/total_fact

In [51]:
skus_analysis.columns

Index(['COD_ARTICULO', 'ARTICULO_gral', 'NUMERO_FACTURA_gral', 'weight_gral',
       'ARTICULO_alone', 'NUMERO_FACTURA_alone', 'weight_alone',
       'weight_overeach', 'weight_overall'],
      dtype='object')

In [52]:
skus_analysis = skus_analysis[[
    'COD_ARTICULO', 'ARTICULO_gral', 
    'NUMERO_FACTURA_gral', 'NUMERO_FACTURA_alone', 
    'weight_gral','weight_alone',
    'weight_overeach', 'weight_overall'
]].copy()

NRO_FACTURAS_gral: número total de facturas en las que el SKU aparece al menos una vez, independientemente de si la factura contiene otros SKUs.

NRO_FACTURAS_alone: número de facturas en las que el SKU aparece de forma exclusiva, es decir, facturas que contienen únicamente ese SKU y ningún otro.

weight_gral: proporción de facturas en las que aparece el SKU respecto al total de facturas emitidas. Mide la presencia general del SKU en el conjunto completo de facturación.

weight_alone: proporción de facturas de un solo SKU en las que aparece el SKU, respecto al total de facturas que contienen únicamente un SKU. Refleja la participación del SKU dentro de las facturas unitarias.

weight_overeach: proporción de facturas en las que el SKU aparece solo respecto al total de facturas en las que dicho SKU aparece al menos una vez. Indica qué tan frecuentemente el SKU se vende de manera exclusiva cuando está presente en una factura.

weight_overall: proporción de facturas en las que el SKU aparece solo respecto al total de facturas emitidas. Representa el peso absoluto de las ventas exclusivas del SKU sobre toda la facturación.

In [54]:
skus_analysis = skus_analysis[skus_analysis['NUMERO_FACTURA_alone']>=300]

In [56]:
skus_analysis

,COD_ARTICULO,ARTICULO_gral,NUMERO_FACTURA_gral,NUMERO_FACTURA_alone,weight_gral,weight_alone,weight_overeach,weight_overall
0,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,4920,995.0,0.017919,0.007160,0.202236,0.003624
1,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,4196,1557.0,0.015282,0.011204,0.371068,0.005671
2,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),3868,915.0,0.014088,0.006584,0.236556,0.003333
3,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,3828,623.0,0.013942,0.004483,0.162748,0.002269
4,751353,VITAMINA C MULTISABOR 60MG X 320 COMP GENERICO LI,3754,462.0,0.013673,0.003325,0.123069,0.001683
5,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),3471,1578.0,0.012642,0.011355,0.454624,0.005747
6,745391,VITAMINA C MULTISABOR 100MG X 500 COMP GENERICO,3001,344.0,0.010930,0.002475,0.114628,0.001253
7,250937,REFRIANEX X 500 COMP (ANTIGRIPAL) V+,2884,681.0,0.010504,0.004900,0.236130,0.002480
8,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,2678,532.0,0.009754,0.003828,0.198656,0.001938
9,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,2606,497.0,0.009491,0.003576,0.190714,0.001810


In [57]:
skus_analysis = skus_analysis.sort_values(
    by=['weight_gral'], 
    ascending=[False]).round(5)

In [ ]:
#skus_analysis.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\analysis_sku.xlsx", index=False)

In [58]:
ventas_farmacia.columns

Index(['CAT 0', 'CAT 1', 'CAT 2', 'CAT 3', 'CAT 4', 'COD_ARTICULO', 'ARTICULO',
       'Precio Unitario FA', 'CR Unitario', 'FECHA_FACT', 'COD_BODEGA',
       'NUMERO_FACTURA', 'UNIDADES', 'VENTA_NETA', 'BODEGA', 'UNE', 'BODEGA2',
       'CIUDAD', 'REGION', 'FORMATO', 'NSE', 'CLUSTER', 'BEAUTY',
       'HOSPITALARIA'],
      dtype='object')

### Searching the AyB best pairs for top 500

In [59]:
# selfmerge for finding pairs of CAT 4 on same invoice
pairs_factura = pd.merge(
    ventas_farmacia,
    ventas_farmacia,
    on=["NUMERO_FACTURA"],
    suffixes=("_A", "_B")
)

In [60]:
pairs_factura.columns

Index(['CAT 0_A', 'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A',
       'ARTICULO_A', 'Precio Unitario FA_A', 'CR Unitario_A', 'FECHA_FACT_A',
       'COD_BODEGA_A', 'NUMERO_FACTURA', 'UNIDADES_A', 'VENTA_NETA_A',
       'BODEGA_A', 'UNE_A', 'BODEGA2_A', 'CIUDAD_A', 'REGION_A', 'FORMATO_A',
       'NSE_A', 'CLUSTER_A', 'BEAUTY_A', 'HOSPITALARIA_A', 'CAT 0_B',
       'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B',
       'ARTICULO_B', 'Precio Unitario FA_B', 'CR Unitario_B', 'FECHA_FACT_B',
       'COD_BODEGA_B', 'UNIDADES_B', 'VENTA_NETA_B', 'BODEGA_B', 'UNE_B',
       'BODEGA2_B', 'CIUDAD_B', 'REGION_B', 'FORMATO_B', 'NSE_B', 'CLUSTER_B',
       'BEAUTY_B', 'HOSPITALARIA_B'],
      dtype='object')

In [61]:
# filtering combinations to avoid duplicates
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] != pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["ARTICULO_A"] > pairs_factura["ARTICULO_B"]]

# keeping only one order of each pair
pairs_factura = pairs_factura[pairs_factura["CAT 4_A"] != pairs_factura["CAT 4_B"]]


In [62]:
ventasABxfac = pairs_factura.groupby([
    'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A',
    'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B'
]
    ).agg({'NUMERO_FACTURA':'nunique'}).reset_index()

In [63]:
ventasABxfac = ventasABxfac.rename(columns={'NUMERO_FACTURA' : 'NUMERO_FACTURA_AB'})

In [64]:
ventasABxfac.sort_values(by=['NUMERO_FACTURA_AB'], ascending=[False])

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_AB
235874,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,VITAMINAS Y MINERALES,VITAMINAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,687
278733,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS DESCARTABLES,INSUMOS DESCARATABLES DE USO GENERAL,106421,JERINGA DESC 10ML C/AG 21 X 1 1/2 FARMACORP<H...,ETICOS AGUDOS,HORMONAS,CORTICOSTEROIDES VIA GENERAL(CORTICOIDES),CORTICOSTEROIDES SOLOS,751140,DEXAMETASONA 8MG IM-IV X 100 AMP/2ML GENERICO LI,323
235180,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIDIARREICOS,RESTAURADOR ELECTROLITOS ORAL,12123,CURADIL 90 X 250ML SUERO ORAL REHIDRATANTE,232
382829,VITAMINAS Y MINERALES,VITAMINAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770108751489,ALIA2 X 30 SOBRES EFERVESCENTE,227
251169,GASTRICO,APARATO DIGESTIVO Y METABOLICO,HEPATOPROTECTORES,HEPATOPROTECTORES,7862117781158,HEPALIVE FORTE X 40 CAP HEPATOPROTECTOR,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIDIARREICOS,RESTAURADOR ELECTROLITOS ORAL,12123,CURADIL 90 X 250ML SUERO ORAL REHIDRATANTE,214
...,...,...,...,...,...,...,...,...,...,...,...,...,...
394198,VITAMINAS Y MINERALES,VITAMINAS PEDIATRICAS,OMEGA 3 GOMITAS,OMEGA 3 GOMITAS,817432010909,GUMYS OMEGA 3 X 60 GOMITA MASTICABLE(TERBONOVA),VITAMINAS Y MINERALES,SUPLEMENTO DE NUTRICION DEPORTIVA MP,BARRA DE PROTEINA,BARRA DE PROTEINA,7898225524110,ATH BEST WHEY 15G PROTEIN CHOCOLATE CROC X 62GR,1
394199,VITAMINAS Y MINERALES,VITAMINAS PEDIATRICAS,OMEGA 3 GOMITAS,OMEGA 3 GOMITAS,817432010909,GUMYS OMEGA 3 X 60 GOMITA MASTICABLE(TERBONOVA),VITAMINAS Y MINERALES,SUPLEMENTO DE NUTRICION DEPORTIVA MP,SNACK DE PROTEINA,SNACK DE PROTEINA,7899621106306,ATH BEST WHEY PROTEIN BALL DARK MILK CRUNCH X ...,1
394200,VITAMINAS Y MINERALES,VITAMINAS PEDIATRICAS,OMEGA 3 GOMITAS,OMEGA 3 GOMITAS,817432010909,GUMYS OMEGA 3 X 60 GOMITA MASTICABLE(TERBONOVA),VITAMINAS Y MINERALES,VITAMINAS,MULTIVITAMINAS PARA ADULTOS,MULTIVITAMINAS PARA ADULTOS,817432010251,FULL SPECTRUM CJA X 30 TAB (TERBONOVA),1
394201,VITAMINAS Y MINERALES,VITAMINAS PEDIATRICAS,OMEGA 3 GOMITAS,OMEGA 3 GOMITAS,817432010909,GUMYS OMEGA 3 X 60 GOMITA MASTICABLE(TERBONOVA),VITAMINAS Y MINERALES,VITAMINAS,VITAMINA C,VITAMINA C,125601,CEBION C 100MG GOTAS X 30ML VITAMINA C,1


In [65]:
ventasABxfac['weight_AB'] = ventasABxfac['NUMERO_FACTURA_AB'] / total_fact

In [66]:
ventasxpairs = pd.merge(
    ventasABxfac,
    ventas_sku,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO'
)

In [68]:
ventasxpairs_final = pd.merge(
    ventasxpairs[[
        'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
        'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
        'NUMERO_FACTURA_AB', 'weight_AB', 'NUMERO_FACTURA', 'weight']],
    ventas_sku,
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO'
)

In [69]:
ventasxpairs_final = ventasxpairs_final.rename(columns={'weight_x':'weight_A',
                                                        'NUMERO_FACTURA_x':'NUMERO_FACTURA_A',
                                                        'weight_y':'weight_B',
                                                        'NUMERO_FACTURA_y':'NUMERO_FACTURA_B'})

In [70]:
ventasxpairs_final = ventasxpairs_final[[
    'CAT 1_A', 'CAT 2_A', 'CAT 3_A', 'CAT 4_A', 'COD_ARTICULO_A', 'ARTICULO_A', 
    'CAT 1_B', 'CAT 2_B', 'CAT 3_B', 'CAT 4_B', 'COD_ARTICULO_B', 'ARTICULO_B', 
    'NUMERO_FACTURA_A', 'weight_A', 
    'NUMERO_FACTURA_B', 'weight_B',
    'NUMERO_FACTURA_AB', 'weight_AB'
]]

In [71]:
ventasxpairs_final = ventasxpairs_final.sort_values(
    by=['COD_ARTICULO_A',
        'NUMERO_FACTURA_AB','weight_AB'], 
        ascending=[False, False, False])

In [72]:
ventasxpairs_final

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_A,weight_A,NUMERO_FACTURA_B,weight_B,NUMERO_FACTURA_AB,weight_AB
79634,ETICOS AGUDOS,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERIANOS MACROLIDOS Y SIMILARES,UPJOHN-00003,DALACIN C 300MG X 48 CAPS CLINDAMICINA CLORHID...,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,8,0.000029,2606,0.009491,1,0.000004
79635,ETICOS AGUDOS,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERIANOS MACROLIDOS Y SIMILARES,UPJOHN-00003,DALACIN C 300MG X 48 CAPS CLINDAMICINA CLORHID...,ETICOS TRATAMIENTO,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERINOS FLUORQUINOLONAS,7770102001382,BACTIFREN 500MG X 7 COMP LEVOFLOXACINO,8,0.000029,57,0.000208,1,0.000004
251661,GASTRICO,APARATO DIGESTIVO Y METABOLICO,LAXANTES,EMOLIENTES,TELCHI-00135,VASELINA LIQUIDA FCO X 250ML <GLN>,AGUAS-ISOTONICOS Y ENERGIZANTES,HIDRATANTES,ISOTONICOS,ISOTONICOS SIN AZUCAR,650240063220,SUEROX BEBIDA HIDRATANTE X 630ML MANZANA,30,0.000109,342,0.001246,1,0.000004
251662,GASTRICO,APARATO DIGESTIVO Y METABOLICO,LAXANTES,EMOLIENTES,TELCHI-00135,VASELINA LIQUIDA FCO X 250ML <GLN>,AGUAS-ISOTONICOS Y ENERGIZANTES,HIDRATANTES,ISOTONICOS,ISOTONICOS SIN AZUCAR,650240063237,SUEROX BEBIDA HIDR. X 630ML NARANJA-MANDARINA,30,0.000109,312,0.001136,1,0.000004
251663,GASTRICO,APARATO DIGESTIVO Y METABOLICO,LAXANTES,EMOLIENTES,TELCHI-00135,VASELINA LIQUIDA FCO X 250ML <GLN>,AGUAS-ISOTONICOS Y ENERGIZANTES,HIDRATANTES,ISOTONICOS,ISOTONICOS SIN AZUCAR,650240063244,SUEROX BEBIDA HIDR. X 630ML MORA AZUL-HIERBABUENA,30,0.000109,706,0.002571,1,0.000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361374,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,CUIDADO PERSONAL VARIOS,ACCESORIOS PARA EL CABELLO,LIGAS Y HORQUILLAS,HORQUILLAS,041457034828,GOODY SIMPLE SPIN PIN MINI 3 PZA 03482<DES,2,0.000007,1,0.000004,1,0.000004
361375,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,CUIDADO PERSONAL VARIOS,DEPILACION Y AFEITADO,AFEITADO,MAQUINAS DE AFEITAR,7702018037803,GILLETTE MACH3 MAQUINA SENSITIVE<DESC>,2,0.000007,4,0.000015,1,0.000004
361376,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS MEDICOS/VARIOS,INSUMOS MEDICOS/VARIOS,106058000075,GEL PACK REFRIGERA P/VACUNAS FARMACORP X90GR(F...,2,0.000007,804,0.002928,1,0.000004
361377,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,LABIOS,BRILLOS DE LABIOS,4059729422323,ESSENCE ESMALTE FRENCH MANICURE SHEER BEAUTY 0...,2,0.000007,16,0.000058,1,0.000004


In [73]:
top_sku_pairs = ventasxpairs_final.groupby('COD_ARTICULO_A').head(10).reset_index(drop=True)

In [74]:
top_sku_pairs

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_A,weight_A,NUMERO_FACTURA_B,weight_B,NUMERO_FACTURA_AB,weight_AB
0,ETICOS AGUDOS,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERIANOS MACROLIDOS Y SIMILARES,UPJOHN-00003,DALACIN C 300MG X 48 CAPS CLINDAMICINA CLORHID...,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,8,0.000029,2606,0.009491,1,0.000004
1,ETICOS AGUDOS,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERIANOS MACROLIDOS Y SIMILARES,UPJOHN-00003,DALACIN C 300MG X 48 CAPS CLINDAMICINA CLORHID...,ETICOS TRATAMIENTO,ANTIINFECCIOSOS VIA GENERAL,ANTIBACTERIANOS SISTEMICOS,ANTIBACTERINOS FLUORQUINOLONAS,7770102001382,BACTIFREN 500MG X 7 COMP LEVOFLOXACINO,8,0.000029,57,0.000208,1,0.000004
2,GASTRICO,APARATO DIGESTIVO Y METABOLICO,LAXANTES,EMOLIENTES,TELCHI-00135,VASELINA LIQUIDA FCO X 250ML <GLN>,AGUAS-ISOTONICOS Y ENERGIZANTES,HIDRATANTES,ISOTONICOS,ISOTONICOS SIN AZUCAR,650240063220,SUEROX BEBIDA HIDRATANTE X 630ML MANZANA,30,0.000109,342,0.001246,1,0.000004
3,GASTRICO,APARATO DIGESTIVO Y METABOLICO,LAXANTES,EMOLIENTES,TELCHI-00135,VASELINA LIQUIDA FCO X 250ML <GLN>,AGUAS-ISOTONICOS Y ENERGIZANTES,HIDRATANTES,ISOTONICOS,ISOTONICOS SIN AZUCAR,650240063237,SUEROX BEBIDA HIDR. X 630ML NARANJA-MANDARINA,30,0.000109,312,0.001136,1,0.000004
4,GASTRICO,APARATO DIGESTIVO Y METABOLICO,LAXANTES,EMOLIENTES,TELCHI-00135,VASELINA LIQUIDA FCO X 250ML <GLN>,AGUAS-ISOTONICOS Y ENERGIZANTES,HIDRATANTES,ISOTONICOS,ISOTONICOS SIN AZUCAR,650240063244,SUEROX BEBIDA HIDR. X 630ML MORA AZUL-HIERBABUENA,30,0.000109,706,0.002571,1,0.000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74985,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,CUIDADO PERSONAL VARIOS,ACCESORIOS PARA EL CABELLO,LIGAS Y HORQUILLAS,HORQUILLAS,041457034828,GOODY SIMPLE SPIN PIN MINI 3 PZA 03482<DES,2,0.000007,1,0.000004,1,0.000004
74986,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,CUIDADO PERSONAL VARIOS,DEPILACION Y AFEITADO,AFEITADO,MAQUINAS DE AFEITAR,7702018037803,GILLETTE MACH3 MAQUINA SENSITIVE<DESC>,2,0.000007,4,0.000015,1,0.000004
74987,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,INSUMOS MEDICOS,OTROS INSUMOS MEDICOS,INSUMOS MEDICOS/VARIOS,INSUMOS MEDICOS/VARIOS,106058000075,GEL PACK REFRIGERA P/VACUNAS FARMACORP X90GR(F...,2,0.000007,804,0.002928,1,0.000004
74988,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,ROSTRO,PRIMERS,0041554197044,MB MASCARA COLOSSAL X 8ML 7X MAS VOLUMEN,SKIN CARE Y MAQUILLAJE,MAQUILLAJE,LABIOS,BRILLOS DE LABIOS,4059729422323,ESSENCE ESMALTE FRENCH MANICURE SHEER BEAUTY 0...,2,0.000007,16,0.000058,1,0.000004


### Getting the final base for analysis

In [75]:
base_analysis = pd.merge(
    top_sku_pairs,
    skus_analysis,
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='right')

In [76]:
skus_analysis

,COD_ARTICULO,ARTICULO_gral,NUMERO_FACTURA_gral,NUMERO_FACTURA_alone,weight_gral,weight_alone,weight_overeach,weight_overall
0,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,4920,995.0,0.01792,0.00716,0.20224,0.00362
1,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,4196,1557.0,0.01528,0.01120,0.37107,0.00567
2,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),3868,915.0,0.01409,0.00658,0.23656,0.00333
3,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,3828,623.0,0.01394,0.00448,0.16275,0.00227
4,751353,VITAMINA C MULTISABOR 60MG X 320 COMP GENERICO LI,3754,462.0,0.01367,0.00332,0.12307,0.00168
5,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),3471,1578.0,0.01264,0.01136,0.45462,0.00575
6,745391,VITAMINA C MULTISABOR 100MG X 500 COMP GENERICO,3001,344.0,0.01093,0.00248,0.11463,0.00125
7,250937,REFRIANEX X 500 COMP (ANTIGRIPAL) V+,2884,681.0,0.01050,0.00490,0.23613,0.00248
8,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,2678,532.0,0.00975,0.00383,0.19866,0.00194
9,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,2606,497.0,0.00949,0.00358,0.19071,0.00181


In [77]:
base = pd.merge(
    base_analysis,
    cubo[['COD_ARTICULO', 'Precio Unitario FA', 'CR Unitario']],
    left_on='COD_ARTICULO_B',
    right_on='COD_ARTICULO',
    how='left'
)

In [78]:
base_cat1 = pd.merge(
    base,
    cubo[['COD_ARTICULO', 'CAT 1']],
    left_on='COD_ARTICULO_A',
    right_on='COD_ARTICULO',
    how='left'
)

In [79]:
base_cat1

,CAT 1_A,CAT 2_A,CAT 3_A,CAT 4_A,COD_ARTICULO_A,ARTICULO_A,CAT 1_B,CAT 2_B,CAT 3_B,CAT 4_B,...,NUMERO_FACTURA_alone,weight_gral,weight_alone,weight_overeach,weight_overall,COD_ARTICULO_y,Precio Unitario FA,CR Unitario,COD_ARTICULO,CAT 1
0,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,VITAMINAS Y MINERALES,VITAMINAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,...,995.0,0.01792,0.00716,0.20224,0.00362,7770105009866,6.50,4.808300,7770101007873,GASTRICO
1,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIDIARREICOS,RESTAURADOR ELECTROLITOS ORAL,...,995.0,0.01792,0.00716,0.20224,0.00362,12123,25.80,20.000000,7770101007873,GASTRICO
2,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,VITAMINAS Y MINERALES,VITAMINAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,OTRAS VITAMINAS SOLAS Y COMBINADAS,...,995.0,0.01792,0.00716,0.20224,0.00362,7770101006784,7.00,5.053276,7770101007873,GASTRICO
3,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,GASTRICO,APARATO DIGESTIVO Y METABOLICO,DIGESTIVOS INCL.ENZIMAS,DIGESTIVOS INCL.ENZIMAS,...,995.0,0.01792,0.00716,0.20224,0.00362,7770105009149,5.20,3.818368,7770101007873,GASTRICO
4,GASTRICO,APARATO DIGESTIVO Y METABOLICO,ANTIACIDOS ANTIFLATULENTOS,ANTIACIDOS SOLOS,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,GASTRICO,APARATO DIGESTIVO Y METABOLICO,HEPATOPROTECTORES,HEPATOPROTECTORES,...,995.0,0.01792,0.00716,0.20224,0.00362,7862117781158,5.48,3.581598,7770101007873,GASTRICO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507,ETICOS AGUDOS,PRODUCTOS GENITO URINARIOS,UROLOGICOS,OTRAS PREPARACIONES UROLOGICAS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,ANTIARLEGICOS Y RESPIRATORIOS,APARATO RESPIRATORIO,ANTIHISTAMINICOS,ANTIHISTAMINICOS-ANTIALERGICOS,...,347.0,0.00203,0.00250,0.62186,0.00126,18022,3.75,2.739000,7840653009103,ETICOS AGUDOS
508,ETICOS AGUDOS,PRODUCTOS GENITO URINARIOS,UROLOGICOS,OTRAS PREPARACIONES UROLOGICAS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,...,347.0,0.00203,0.00250,0.62186,0.00126,7770108268673,8.41,6.175200,7840653009103,ETICOS AGUDOS
509,ETICOS AGUDOS,PRODUCTOS GENITO URINARIOS,UROLOGICOS,OTRAS PREPARACIONES UROLOGICAS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,ETICOS AGUDOS,APARATO LOCOMOTOR,ANTIINFLAMATORIOS Y ANTIRREUMATICOS,ANTIRREUMATICOS NO ESTEROIDEOS,...,347.0,0.00203,0.00250,0.62186,0.00126,7793640215523,5.33,3.861471,7840653009103,ETICOS AGUDOS
510,ETICOS AGUDOS,PRODUCTOS GENITO URINARIOS,UROLOGICOS,OTRAS PREPARACIONES UROLOGICAS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,RESFRIO/DOLOR,SISTEMA NERVIOSO CENTRAL,ANALGESICOS,ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.,...,347.0,0.00203,0.00250,0.62186,0.00126,7800063111438,0.61,0.317700,7840653009103,ETICOS AGUDOS


In [80]:
base_cat1[['CAT 4_A', 'CAT 4_B']].value_counts()

CAT 4_A                                   CAT 4_B                                    
ANTIGRIPALES EXC.ANTIINFARINGITIS         ANTIRREUMATICOS NO ESTEROIDEOS                 16
GASEOSAS                                  AGUA SIN GAS                                   12
ANTIRREUMATICOS NO ESTEROIDEOS            ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.        9
ANALGESICOS NO NARCOTICOS ANTIPIRETICOS.  ANTIRREUMATICOS NO ESTEROIDEOS                  8
ANTIRREUMATICOS NO ESTEROIDEOS            ANTIBACTERIANOS PENICILINAS AMPLIO ESPECTRO     8
                                                                                         ..
OTRAS PREPARACIONES UROLOGICAS            GEL LUBRICANTE                                  1
OTROS PRODUCTOS GINECOLOGICOS             ANTIHIPERTENSIVOS/INHIBIDORES ECA SOLOS         1
VITAMINA C                                ANTIULCEROSOS                                   1
                                          DIGESTIVOS INCL.ENZIMAS                     

In [82]:
base_final = base_cat1[[
    'CAT 1',
    'COD_ARTICULO_A', 'ARTICULO_A', 
    'COD_ARTICULO_B', 'ARTICULO_B',
    
    'NUMERO_FACTURA_A', 'weight_A',
    'NUMERO_FACTURA_B', 'weight_B',
    'NUMERO_FACTURA_AB', 'weight_AB',
    
    'NUMERO_FACTURA_alone', 'weight_alone', 
    'weight_overeach', 'weight_overall',
    
    'Precio Unitario FA',
    'CR Unitario'
]]

In [83]:
base_final = base_final.rename(columns={
    'Precio Unitario FA': 'PVU_B',
    'CR Unitario' : 'CRU_B'
    })

In [84]:
base_final['Profit_B'] = base_final['PVU_B'] - base_final['CRU_B']

In [89]:
base_final

,CAT 1,COD_ARTICULO_A,ARTICULO_A,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_A,weight_A,NUMERO_FACTURA_B,weight_B,NUMERO_FACTURA_AB,weight_AB,NUMERO_FACTURA_alone,weight_alone,weight_overeach,weight_overall,PVU_B,CRU_B,Profit_B,Profit_B_avg,ambition
0,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,4920.0,0.017919,3828.0,0.013942,687.0,0.002502,995.0,0.00716,0.20224,0.00362,6.50,4.808300,1.691700,2.550723,0.05
1,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,12123,CURADIL 90 X 250ML SUERO ORAL REHIDRATANTE,4920.0,0.017919,1589.0,0.005787,232.0,0.000845,995.0,0.00716,0.20224,0.00362,25.80,20.000000,5.800000,2.550723,0.05
2,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7770101006784,DEXTROTON CJA X 36 SOBRES,4920.0,0.017919,935.0,0.003405,176.0,0.000641,995.0,0.00716,0.20224,0.00362,7.00,5.053276,1.946724,2.550723,0.05
3,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,4920.0,0.017919,2678.0,0.009754,172.0,0.000626,995.0,0.00716,0.20224,0.00362,5.20,3.818368,1.381632,2.550723,0.05
4,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7862117781158,HEPALIVE FORTE X 40 CAP HEPATOPROTECTOR,4920.0,0.017919,1144.0,0.004167,160.0,0.000583,995.0,0.00716,0.20224,0.00362,5.48,3.581598,1.898402,2.550723,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,18022,ALERGIN 4MG X 48 COMP CLORFENIRAMINA MALEATO,558.0,0.002032,1299.0,0.004731,3.0,0.000011,347.0,0.00250,0.62186,0.00126,3.75,2.739000,1.011000,3.256046,0.05
506,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,7770108268673,NOVADOL 75/500MG X 120 CAP DICLOFENACO/PARACET...,558.0,0.002032,2007.0,0.007310,3.0,0.000011,347.0,0.00250,0.62186,0.00126,8.41,6.175200,2.234800,3.256046,0.05
507,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,558.0,0.002032,2606.0,0.009491,3.0,0.000011,347.0,0.00250,0.62186,0.00126,5.33,3.861471,1.468529,3.256046,0.05
508,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,7800063111438,PARACETAMOL 500MG X 96 COMP (MLAB)<DESC>,558.0,0.002032,454.0,0.001654,3.0,0.000011,347.0,0.00250,0.62186,0.00126,0.61,0.317700,0.292300,3.256046,0.05


In [86]:
profit_top_sku = (
    base_final
    .groupby('COD_ARTICULO_A', as_index=False)['Profit_B']
    .mean()
    .rename(columns={'Profit_B': 'Profit_B_avg'})
)

In [87]:
base_final = pd.merge(
    base_final,
    profit_top_sku,
    on='COD_ARTICULO_A'
)

In [91]:
base_final['ambition'] = 0.05

In [93]:
base_final['opportunity'] = base_final['ambition'] * base_final['NUMERO_FACTURA_alone'] * base_final['Profit_B']

In [94]:
base_final['opportunity_avg'] = base_final['ambition'] * base_final['NUMERO_FACTURA_alone'] * base_final['Profit_B_avg']

In [95]:
base_final

,CAT 1,COD_ARTICULO_A,ARTICULO_A,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_A,weight_A,NUMERO_FACTURA_B,weight_B,NUMERO_FACTURA_AB,...,weight_alone,weight_overeach,weight_overall,PVU_B,CRU_B,Profit_B,Profit_B_avg,ambition,opportunity,opportunity_avg
0,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,4920.0,0.017919,3828.0,0.013942,687.0,...,0.00716,0.20224,0.00362,6.50,4.808300,1.691700,2.550723,0.05,84.162075,126.898486
1,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,12123,CURADIL 90 X 250ML SUERO ORAL REHIDRATANTE,4920.0,0.017919,1589.0,0.005787,232.0,...,0.00716,0.20224,0.00362,25.80,20.000000,5.800000,2.550723,0.05,288.550000,126.898486
2,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7770101006784,DEXTROTON CJA X 36 SOBRES,4920.0,0.017919,935.0,0.003405,176.0,...,0.00716,0.20224,0.00362,7.00,5.053276,1.946724,2.550723,0.05,96.849524,126.898486
3,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,4920.0,0.017919,2678.0,0.009754,172.0,...,0.00716,0.20224,0.00362,5.20,3.818368,1.381632,2.550723,0.05,68.736202,126.898486
4,GASTRICO,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,7862117781158,HEPALIVE FORTE X 40 CAP HEPATOPROTECTOR,4920.0,0.017919,1144.0,0.004167,160.0,...,0.00716,0.20224,0.00362,5.48,3.581598,1.898402,2.550723,0.05,94.445514,126.898486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,18022,ALERGIN 4MG X 48 COMP CLORFENIRAMINA MALEATO,558.0,0.002032,1299.0,0.004731,3.0,...,0.00250,0.62186,0.00126,3.75,2.739000,1.011000,3.256046,0.05,17.540850,56.492404
506,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,7770108268673,NOVADOL 75/500MG X 120 CAP DICLOFENACO/PARACET...,558.0,0.002032,2007.0,0.007310,3.0,...,0.00250,0.62186,0.00126,8.41,6.175200,2.234800,3.256046,0.05,38.773780,56.492404
507,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,558.0,0.002032,2606.0,0.009491,3.0,...,0.00250,0.62186,0.00126,5.33,3.861471,1.468529,3.256046,0.05,25.478973,56.492404
508,ETICOS AGUDOS,7840653009103,PROCOPS 100MG X 10 COMP SILDENAFIL,7800063111438,PARACETAMOL 500MG X 96 COMP (MLAB)<DESC>,558.0,0.002032,454.0,0.001654,3.0,...,0.00250,0.62186,0.00126,0.61,0.317700,0.292300,3.256046,0.05,5.071405,56.492404


In [96]:
base_final.to_excel(r"C:\Users\fdavila\OneDrive - Farmacorp S.A\Escritorio\ANALISIS -CORRELACION\output\parejas_OTC_Dias01-05Marzo.xlsx",
                     index=False)

In [98]:
# Top 50 ARTICULO_A by NRO_FACTURAS_A
if 'NUMERO_FACTURA_A' not in base.columns:
    raise KeyError("'NUMERO_FACTURA_A' not found in `base`. Check previous merges.")

group_cols = ['COD_ARTICULO_A', 'ARTICULO_A']

top50_articulo_A = (
    base
    .groupby(group_cols, as_index=False)['NUMERO_FACTURA_A']
    .max()
    .sort_values(by='NUMERO_FACTURA_A', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# attach weight_A if available
if 'weight_A' in base.columns:
    top50_articulo_A = top50_articulo_A.merge(
        base.groupby(group_cols, as_index=False)['weight_A'].max(),
        on=group_cols,
        how='left'
    )


In [99]:

# display
top50_articulo_A

,COD_ARTICULO_A,ARTICULO_A,NUMERO_FACTURA_A,weight_A
0,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,4920.0,0.017919
1,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,4196.0,0.015282
2,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),3868.0,0.014088
3,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,3828.0,0.013942
4,751353,VITAMINA C MULTISABOR 60MG X 320 COMP GENERICO LI,3754.0,0.013673
5,7703763995486,ZOPICLONA 7.5MG X 30 TAB (LA SANTE),3471.0,0.012642
6,745391,VITAMINA C MULTISABOR 100MG X 500 COMP GENERICO,3001.0,0.010930
7,250937,REFRIANEX X 500 COMP (ANTIGRIPAL) V+,2884.0,0.010504
8,7770105009149,DIGESTAN COMPUESTO X 50 SOBRES,2678.0,0.009754
9,7793640215523,ACTRON 600MG X 10 CAP BLANDAS IBUPROFENO,2606.0,0.009491


In [103]:
# Top 50 ARTICULO_B by NRO_FACTURAS_B (including partners count)
if 'NUMERO_FACTURA_B' not in base.columns:
    raise KeyError("'NUMERO_FACTURA_B' not found in `base`. Check previous merges.")

group_cols_b = ['COD_ARTICULO_B', 'ARTICULO_B']

# aggregate invoices per ARTICULO_B
agg_b = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(NUMERO_FACTURA_B=('NUMERO_FACTURA_B', 'max'))
)

# count distinct partners (COD_ARTICULO_A) per ARTICULO_B
partners = (
    base
    .groupby(group_cols_b, as_index=False)
    .agg(partners_count=('COD_ARTICULO_A', 'nunique'))
)

# merge results
top50_articulo_B = agg_b.merge(partners, on=group_cols_b, how='left')

# attach weight_B if available
if 'weight_B' in base.columns:
    w = base.groupby(group_cols_b, as_index=False)['weight_B'].max()
    top50_articulo_B = top50_articulo_B.merge(w, on=group_cols_b, how='left')

# sort and take top 50
top50_articulo_B = (
    top50_articulo_B
    .sort_values(by='NUMERO_FACTURA_B', ascending=False)
    .head(100)
    .reset_index(drop=True)
)


In [104]:
# display
top50_articulo_B

,COD_ARTICULO_B,ARTICULO_B,NUMERO_FACTURA_B,partners_count,weight_B
0,7770101007873,RESSAKA X 60 SOBRES EFERVECENTE S/GUARANA,4920.0,6,0.017919
1,7770101005930,QUETOROL SL 30MG X 10 TAB SUBL KETOROLACO,4196.0,9,0.015282
2,7770108121671,TYPIREC X 200 CAP BLANDA (ANTIGRIPAL),3868.0,2,0.014088
3,7770105009866,GLUCOSAMIN 12 X 36 SOBRES,3828.0,19,0.013942
4,250937,REFRIANEX X 500 COMP (ANTIGRIPAL) V+,2884.0,3,0.010504
...,...,...,...,...,...
95,7800063910284,FUROSEMIDA 40MG X 12 COMP (MLAB),472.0,1,0.001719
96,7770102000576,BIL 13 X 150 COMP V+,472.0,1,0.001719
97,7750215025086,BISMUTIP 262.5MG X 50 SOBRES/2 TAB MAST BISMUT...,472.0,1,0.001719
98,7771259756392,CHICOLAC LECHE CHOCOLATADA X 120ML,471.0,1,0.001715
